# Prepare GtR data for Mission Radar

In [2]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import gtr
from discovery_utils.utils.io import safe_yaml_load

from discovery_utils.utils.llm import batch_check

from discovery_utils.utils import keywords as kw

PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [3]:
from datetime import datetime

def convert_to_date(x: str) -> str:
    try:
        return datetime.fromtimestamp(x / 1000).strftime('%Y-%m-%d')
    except:
        return ""

In [4]:
GTR = gtr.GtrGetter(data_version = "GtR_20250309")
# GTR = gtr.GtrGetter()

In [5]:
CONFIG_NAMES = [
    # "bioenergy",
    "biomass_heating",
    # "built_environment",
    "ccus",
    "district_heating",
    "energy_efficiency",
    # "energy_grid",
    "geothermal_energy",
    # "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
    "energy_storage",    
    # "renewables_general",    
    "solar",
    "wind"
    # "decarbonisation_general",    
]

## Run the LLM checks

In [5]:
start_date = "2014-01-01"
end_date = "2025-03-31"

new_projects = (
    GTR.projects_enriched
    .query("(start >= @start_date and start <= @end_date)")
)
new_projects_text = GTR.get_projects_text().query("id in @new_projects.id.to_list()")

2025-04-11 17:43:14,681 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250309/projects.parquet
2025-04-11 17:43:40,047 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250309/projects.parquet
2025-04-11 17:43:41,108 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250309/funds.parquet
2025-04-11 17:43:42,854 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250309/funds.parquet


In [6]:
try:
    enrichment_df = (
        pd.read_csv(OUTPUT_DIR / "gtr_labelled_projects.csv")
        .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",")))
        .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",")))
    )

except FileNotFoundError:
    enrichment_df = kw.enrich_topic_labels(new_projects_text) # will take 5-6 minutes
    enrichment_df.to_csv(OUTPUT_DIR / "gtr_labelled_projects.csv", index=False)

In [12]:
from typing import List, Literal

def get_projects_in_nesta_categories(
    GTR,
    enrichment_df: pd.DataFrame,
    category_type: Literal["mission_labels", "topic_labels"],
    categories: List[str],
) -> pd.DataFrame:
    """Get all companies belonging to the provided categories"""
    matching_ids = (  # noqa
        enrichment_df
        .explode(category_type)
        .query(f"{category_type} in @categories")
        .id.to_list()
    )
    return GTR.projects_enriched.query("id in @matching_ids").drop_duplicates(subset="id")

def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_projects_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return get_projects_in_nesta_categories(GTR, enrichment_df, "topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = GTR.get_projects_text().query("id in @selected_df.id.to_list()")
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
   # system_message = batch_check.generate_relevance_check_system_message(config)

    # fields = [
    #     {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    # ]
    system_message = batch_check.generate_relevance_check_system_message(config)
    system_message += """ 
    Mark the text as 'yes' if (one or more of the following):
    - If the technology defined by the scope above, is the main focus
    - If the technology is one of the components or activities described by the text. For example the technology could be  mentioned
        as part of a larger project or business including other technologies, or be mentioned as one of the use cases or case studies.
    - If the text describes a company and the technology is the main focus, or a part of a broader range of the company's activities and offerings.
    - If the text is about a component or critical element of the technology defined above
    - If the text describes a technology or process and explicitly mentions that it can be applied on the technology defined above to improve it's performance or efficiency.

    If the text is about heating technology but application target is not mentioned then assume it could be relevant for households or buildings (as opposed to an industrial applications).

    However, mark it as 'no' if (one or more of the following):
    - The activities, or business described in the text does not have a discernable impact on or connection with the technology.
    - If the technology is mentioned only in passing or as a minor example in a broader discussion, for example, 
        in only one sentence within a long text with many sentences, or at the very end of a long description.
    - The technology is mentioned only as a negative example (eg "unlike [technology]...")
    - The text mentions heat pumps for heating swimming pools  
    - The text would be better captured by one of the other categories (comma separated) mentioned in this list:  
    Bioenergy (biofuels), Biomass heating, Carbon capture and storage, District heating and heat networks, Energy grid, Geothermal energy, 
    Heat pumps, Hydrogen energy, Hydrogen heating, Micro CHP, Solar thermal heating, Energy storage (batteries), Solar power, Wind power
    """    
            
    fields = [
        {"name": "explanation", "type": "str", "description": "A short, 1-sentence explanation of the answer (max 25 words)."},
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},    
    ]    

    processor = batch_check.LLMProcessor(
        model_name="gpt-4o-mini",
        output_path=str(OUTPUT_DIR / f"gtr_llm_check_v2_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.3)    

In [13]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        config = get_config_dict(config_name)
        selected_df = get_projects_from_config(config)
        logging.info(f"Checking relevance for {config_name} ({len(selected_df)} projects)")
        await check_relevance(selected_df, config_name, config)
        

In [17]:
await check_all_configs(CONFIG_NAMES)

2025-04-12 16:50:34,558 - root - INFO - Checking relevance for biomass_heating (11 projects)
2025-04-12 16:50:35,780 - root - INFO - Using OpenAI
2025-04-12 16:50:35,888 - root - INFO - All data has already been processed.
2025-04-12 16:50:36,058 - root - INFO - Checking relevance for ccus (534 projects)
2025-04-12 16:50:36,817 - root - INFO - Using OpenAI
2025-04-12 16:50:36,875 - root - INFO - All data has already been processed.
2025-04-12 16:50:37,033 - root - INFO - Checking relevance for district_heating (140 projects)
2025-04-12 16:50:37,805 - root - INFO - Using OpenAI
2025-04-12 16:50:37,864 - root - INFO - All data has already been processed.
2025-04-12 16:50:38,027 - root - INFO - Checking relevance for energy_efficiency (1117 projects)
2025-04-12 16:50:38,782 - root - INFO - Using OpenAI
2025-04-12 16:50:38,836 - root - INFO - All data has already been processed.
2025-04-12 16:50:39,006 - root - INFO - Checking relevance for geothermal_energy (180 projects)
2025-04-12 16:50

## Spot check the results

In [ ]:
import pandas as pd
from discovery_utils.utils import google

sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"
tab_name = "ukri_check_v2"

In [ ]:
checked_gtr_df = google.access_google_sheet(sheet_id, "ukri_check")

In [11]:
checked_gtr_df = (
    checked_gtr_df
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
)

In [16]:
ids_already_checked = checked_gtr_df[~checked_gtr_df.reviewer.isna()]

In [18]:
len(ids_already_checked)

80

In [17]:
n_samples = 10

gtr_cols = ["id", "title", "amount", "start", "url"]
final_cols = ["theme", "id", "title", "text", "amount", "start", "url", "model", "temperature", "is_relevant"]

llm_checks_df = []

for config_name in CONFIG_NAMES:
    # read a jsonl file
    llm_check_df = (
        pd.read_json(OUTPUT_DIR / f"gtr_llm_check_{config_name}.jsonl", lines=True)
        .merge(GTR.projects_enriched[gtr_cols], left_on="id", right_on="id", how="left")
        .merge(GTR.get_projects_text()[["id", "text"]], left_on="id", right_on="id", how="left")
        .assign(theme=config_name)
        .query("id not in @ids_already_checked.id.to_list()")
        .groupby("is_relevant")[final_cols]
        # sample n or less from each group
        .apply(lambda df: df.sample(n_samples) if len(df) > n_samples else df)
        .reset_index(drop=True)
    )[final_cols]
    llm_checks_df.append(llm_check_df)

llm_checks_df = pd.concat(llm_checks_df, ignore_index=True)


In [19]:
llm_checks_df

,theme,id,title,text,amount,start,url,model,temperature,is_relevant
0,ccus,2635A7AF-1237-4559-BDAE-C3B42EE75A43,Vital Acoustic Pilot,Vital Acoustic Pilot The UK was the first majo...,49998.0,2024-07-01,https://gtr.ukri.org/projects?ref=10113571,gpt-4o-mini,0,no
1,ccus,86B9243F-85D3-4A8A-8077-23AB6FF77813,Triple whammy - breeding wheat for increased w...,Triple whammy - breeding wheat for increased w...,0.0,2024-10-01,https://gtr.ukri.org/projects?ref=2928085,gpt-4o-mini,0,no
2,ccus,11BD1419-3AD7-4B25-902B-53CD51CCC2E0,University of Leeds Core Equipment Award 2022,University of Leeds Core Equipment Award 2022 ...,1250000.0,2023-01-03,https://gtr.ukri.org/projects?ref=EP/X034801/1,gpt-4o-mini,0,no
3,ccus,F25FFD3C-9010-44D6-85C8-504D8F71ED69,Faults in Coal,Faults in Coal The project will look to addres...,0.0,2016-10-01,https://gtr.ukri.org/projects?ref=1857013,gpt-4o-mini,0,no
4,ccus,2356BC3D-8BDF-4FF0-8DD2-FBF5D842D67C,Modelling the multi-scale spatial variations i...,Modelling the multi-scale spatial variations i...,0.0,2022-09-12,https://gtr.ukri.org/projects?ref=2754950,gpt-4o-mini,0,no
...,...,...,...,...,...,...,...,...,...,...
251,wind,296E699B-200B-492A-9738-92525BAC0686,Cyber Risk-Resilience of Wind Plants: A Formal...,Cyber Risk-Resilience of Wind Plants: A Formal...,0.0,2023-10-01,https://gtr.ukri.org/projects?ref=2881978,gpt-4o-mini,0,yes
252,wind,1ACF2F1F-EEB1-4FF6-8901-A5212C5151F9,Dual-doppler Offshore liDar for optimisAtion o...,Dual-doppler Offshore liDar for optimisAtion o...,295056.0,2024-12-01,https://gtr.ukri.org/projects?ref=10125841,gpt-4o-mini,0,yes
253,wind,D8DE4519-5143-4B7F-929F-9A3531F7273C,Breakdown of helical vortices in wind farms,Breakdown of helical vortices in wind farms Th...,34836.0,2016-11-01,https://gtr.ukri.org/projects?ref=EP/M025039/2,gpt-4o-mini,0,yes
254,wind,CE84001F-44A2-40C3-A43A-F400E25F9AD9,Developing Offshore Floating Technology,Developing Offshore Floating Technology Offsho...,0.0,2022-09-01,https://gtr.ukri.org/projects?ref=2879078,gpt-4o-mini,0,yes


In [20]:
tab_name = "ukri_check_v2"
google.upload_data_to_gsheet(sheet_id, {tab_name: llm_checks_df})
google.format_gsheet(sheet_id, tab_name, freeze_cols=2)

2025-04-24 16:18:30,533 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
2025-04-24 16:18:34,271 - root - INFO - Uploading DataFrame to sheet: ukri_check_v2
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-04-24 16:18:54,014 - root - INFO - Upload completed successfully.
2025-04-24 16:18:55,039 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]


In [ ]:
llm_check_df = (
    pd.read_json(OUTPUT_DIR / f"gtr_llm_check_{config_name}.jsonl", lines=True)
    .merge(GTR.projects_enriched[gtr_cols], left_on="id", right_on="id", how="left")
    .merge(GTR.get_projects_text()[["id", "text"]], left_on="id", right_on="id", how="left")
    .assign(theme=config_name)
    .query("id not in @ids_already_checked.id.to_list()")
    .groupby("is_relevant")[final_cols]
    # sample n or less from each group
    .apply(lambda df: df.sample(n_samples) if len(df) > n_samples else df)
    .reset_index(drop=True)
)[final_cols]
llm_checks_df.append(llm_check_df)